# Notebook 03: GraphRAG Query
## Map-Reduce over Community Summaries

This is where GraphRAG earns its name. We pose a **global sensemaking question**:

> *"What are the main storylines and themes of the 2022 FIFA World Cup?"*

This question has no single correct chunk to retrieve — it requires synthesising information
spread across the entire document. Standard vector RAG will return the three most similar
paragraphs and miss the big picture. GraphRAG answers it by asking each community to
contribute a partial answer, then merging them.

**Paper reference:** Edge et al. (2025) — §3.1.6, Appendix E.3–E.4.

### What this notebook does

```
wc2022_trimmed_summaries.json   (15 community reports)
           │
           ▼  MAP  (App. E.4 — Community Answer Generation)
           │    For each community: LLM scores helpfulness 0–100
           │    and generates a partial answer.
           │    Filter out score = 0.
           │
           ▼  REDUCE  (App. E.4 — Global Answer Generation)
           │    Sort partial answers by score (desc).
           │    LLM synthesises a final markdown answer.
           ▼
       GraphRAG Answer
           │
           ▼  COMPARISON
       Naive RAG Answer  (TF-IDF top-3 chunks → LLM)
```

## Imports and Setup

In [1]:
import json
import re
import subprocess
from pathlib import Path

import fitz
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import Markdown, display

print("Imports OK")

Imports OK


In [2]:
import os
import platform

PDF_STEM      = "wc2022_trimmed"
PDF_PATH      = Path("../data/input")  / f"{PDF_STEM}.pdf"
SUMMARIES_PATH = Path("../data/output") / f"{PDF_STEM}_summaries.json"

QUESTION = "What are the main storylines and themes of the 2022 FIFA World Cup?"


def ollama_base_url() -> str:
    """Locate the Ollama server so this notebook runs on any machine.

    Priority: OLLAMA_BASE_URL env var  →  WSL2 Windows-host IP  →  localhost.
    Set OLLAMA_BASE_URL yourself if Ollama runs somewhere else (e.g. a remote box).
    """
    if os.environ.get("OLLAMA_BASE_URL"):
        return os.environ["OLLAMA_BASE_URL"]
    if "microsoft" in platform.uname().release.lower():   # WSL2 → Ollama on the Windows host
        host = subprocess.check_output(
            "ip route list default | awk '{print $3}'", shell=True
        ).decode().strip()
        if host:
            return f"http://{host}:11434"
    return "http://localhost:11434"                        # native Linux / macOS / Windows

BASE_URL = ollama_base_url()

llm = ChatOllama(
    model="qwen2.5:7b",
    base_url=BASE_URL,
    temperature=0.0,
    num_predict=1500,
)

print(f"Question  : {QUESTION}")
print(f"LLM       : qwen2.5:7b  @  {BASE_URL}")

Question  : What are the main storylines and themes of the 2022 FIFA World Cup?
LLM       : qwen2.5:7b  @  http://172.19.64.1:11434


/tmp/ipykernel_95780/1448428340.py:29: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(


---
## Step 1 — Load Community Summaries

In [3]:
with open(SUMMARIES_PATH, encoding="utf-8") as f:
    summaries = json.load(f)

valid = [s for s in summaries if "_error" not in s and s.get("summary")]
print(f"Loaded {len(summaries)} summaries, {len(valid)} valid.")
print()
for s in valid:
    print(f"  [{s['community_id']}] {s.get('rating', 0):>4.1f} — {s.get('title', '?')[:60]}")

Loaded 15 summaries, 15 valid.

  [2]  8.5 — 2022 FIFA World Cup Final: France vs Argentina
  [3]  8.5 — 2022 FIFA World Cup Qatar - Brazil and Uruguay's Journey
  [8]  8.5 — 2022 FIFA World Cup Community Report
  [4]  7.5 — 2022 FIFA World Cup Group D and Related Entities
  [5]  7.5 — 2022 FIFA World Cup Group B Analysis
  [6]  7.5 — 2022 FIFA World Cup Bidding Process and Related Entities
  [10]  7.5 — Project Merciless and Its Impact on the 2022 World Cup Hosti
  [11]  7.5 — FIFA Corruption Investigation Network
  [0]  6.5 — Qatar's 2022 FIFA World Cup: A Mix of Triumph and Controvers
  [1]  6.5 — FIFA World Cup in Qatar: Community Dynamics and Challenges
  [9]  6.5 — Qatar World Cup Security and Media Relations
  [12]  6.5 — Katara Cultural Village Security Concerns
  [14]  6.5 — Qatar's 2022 World Cup Bid Allegations
  [7]  5.5 — 2026 FIFA World Cup Community Dynamics
  [13]  4.5 — Media-Turf Dispute at 2022 FIFA World Cup


---
## Step 2 — Map: Score Each Community  *(Appendix E.4)*

For every community summary we ask the LLM two things at once:

1. **How helpful is this community for answering the question?** (score 0–100)  
2. **Write a partial answer** based only on this community's content.

The helpfulness score is the key innovation — it lets us automatically rank which
communities are relevant before the Reduce step, without manual filtering.
Communities that score 0 are discarded entirely.

The prompt template (Appendix E.4):

In [4]:
MAP_PROMPT = """\
---Goal---
Generate a response to the user's question based on the community report below.
At the very start of your response, output an integer score 0-100 indicating how
helpful this community report is for answering the question.
Use the format: <ANSWER_HELPFULNESS>score</ANSWER_HELPFULNESS>

Score 0 means the report is not relevant at all.
Score 100 means the report directly and comprehensively answers the question.

---Question---
{question}

---Community Report---
{community_report}

Output:"""


def format_summary(s: dict) -> str:
    """Convert a JSON summary into readable text for the LLM."""
    lines = [
        f"Title: {s.get('title', '')}",
        f"Summary: {s.get('summary', '')}",
        "Key findings:",
    ]
    for f in s.get("findings", []):
        lines.append(f"  - {f.get('summary', '')}: {f.get('explanation', '')}")
    return "\n".join(lines)


def parse_map_response(text: str) -> tuple[int, str]:
    """Extract the helpfulness score and the answer body.

    We anchor on the *opening* tag and read the number right after it, then strip
    the whole annotation. This is deliberately lenient: qwen2.5 often mangles the
    closing tag (e.g. `</ANSWER HELPFULNESS>` with a space instead of an
    underscore), and a strict `<ANSWER_HELPFULNESS>(\\d+)</ANSWER_HELPFULNESS>`
    regex would fail to match and silently score every community 0.
    """
    m = re.search(r'<ANSWER[_ ]?HELPFULNESS>\s*(\d+)', text, re.IGNORECASE)
    score = int(m.group(1)) if m else 0
    answer = re.sub(
        r'<ANSWER[_ ]?HELPFULNESS>\s*\d+\s*(</ANSWER[_ ]?HELPFULNESS>)?',
        '', text, count=1, flags=re.IGNORECASE,
    ).strip()
    return score, answer


def strip_code_fence(text: str) -> str:
    """Unwrap a ```markdown ... ``` fence if the model wrapped its whole answer.

    We ask for markdown output, and qwen2.5 sometimes returns it inside a fenced
    code block — which would then render as a literal code block instead of
    formatted markdown. Only a *full* wrap is removed; inline code fences are kept.
    """
    t = text.strip()
    m = re.match(r'^```[a-zA-Z]*\s*\n(.*?)\n?```\s*$', t, re.DOTALL)
    return m.group(1).strip() if m else t


print("Map helpers defined.")

Map helpers defined.


In [5]:
print(f"Running Map step over {len(valid)} communities ...\n")

map_results = []

for s in valid:
    prompt = MAP_PROMPT.format(
        question=QUESTION,
        community_report=format_summary(s),
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    score, answer = parse_map_response(response.content)
    map_results.append({"community_id": s["community_id"], "score": score, "answer": answer})
    print(f"  [{s['community_id']}] score={score:>3}  {s.get('title','')[:50]}")

print(f"\nMap complete.")

Running Map step over 15 communities ...



  [2] score= 85  2022 FIFA World Cup Final: France vs Argentina


  [3] score= 65  2022 FIFA World Cup Qatar - Brazil and Uruguay's J


  [8] score= 85  2022 FIFA World Cup Community Report


  [4] score= 35  2022 FIFA World Cup Group D and Related Entities


...

  [9] score= 10  Qatar World Cup Security and Media Relations


  [12] score=  0  Katara Cultural Village Security Concerns


  [14] score=  0  Qatar's 2022 World Cup Bid Allegations


  [7] score=  0  2026 FIFA World Cup Community Dynamics


  [13] score=  0  Media-Turf Dispute at 2022 FIFA World Cup

Map complete.


In [6]:
# Filter score=0, sort descending
useful = [r for r in map_results if r["score"] > 0]
useful.sort(key=lambda r: r["score"], reverse=True)

print(f"After filtering score=0: {len(useful)}/{len(map_results)} communities are useful.")
print()
for r in useful:
    print(f"  [{r['community_id']}] score={r['score']:>3}")

After filtering score=0: 9/15 communities are useful.

  [2] score= 85
  [8] score= 85
  [3] score= 65
  [5] score= 60
  [4] score= 35
  [1] score= 35
  [0] score= 20
  [9] score= 10
  [6] score=  5


---
## Step 3 — Reduce: Synthesise a Global Answer  *(Appendix E.4)*

We concatenate the partial answers ranked by helpfulness and ask the LLM to synthesise
them into a single, comprehensive response styled as Markdown.

This is the key insight of GraphRAG: **no single passage in the original text could answer
this question** — the answer emerges from combining community-level knowledge.

The prompt template (Appendix E.4):

In [7]:
REDUCE_PROMPT = """\
---Goal---
Generate a comprehensive response to the question below by synthesising the
analyst reports provided. Style the response in markdown with clear sections.
Do not simply list the reports — integrate them into a coherent narrative.

---Question---
{question}

---Analyst Reports (ranked by helpfulness, highest first)---
{partial_answers}

Output:"""


# Build the ranked partial-answer context
partial_answers_text = "\n\n".join(
    f"[Report {i+1}, relevance score {r['score']}]\n{r['answer']}"
    for i, r in enumerate(useful)
)

reduce_prompt = REDUCE_PROMPT.format(
    question=QUESTION,
    partial_answers=partial_answers_text,
)

print(f"Reduce context: {len(useful)} partial answers, {len(reduce_prompt):,} chars")
print("Running Reduce step ...")

graphrag_response = llm.invoke([HumanMessage(content=reduce_prompt)])
graphrag_answer = strip_code_fence(graphrag_response.content)

print("Done.")

Reduce context: 9 partial answers, 5,716 chars
Running Reduce step ...


Done.


In [8]:
display(Markdown("## GraphRAG Answer\n\n" + graphrag_answer))

## GraphRAG Answer

# Main Storylines and Themes of the 2022 FIFA World Cup

## Overview

The 2022 FIFA World Cup, held in Qatar from November 20 to December 18, 2022, was marked by several key storylines and themes that captured global attention. This report synthesizes insights from various analyst reports to provide a comprehensive overview of the main narratives and themes.

## Key Storylines

### France's Dominance
- **Kylian Mbappé**: The French striker's exceptional goal-scoring record was a central theme, with his performances contributing significantly to France’s success.
- **Golden Ball Award**: Lionel Messi winning the Golden Ball for best player in the tournament highlighted Argentina's triumph and added another layer of narrative.

### Argentina's Triumph
- **Lionel Messi**: His performance and the award he received underscored the emotional journey and legacy of one of football's greatest players.
- **Final Match**: The final match between France and Argentina was a defining moment, with Messi winning the Golden Ball and leading his team to victory.

## Notable Performances

### Senegal's Impressive Journey
- **Enzo Fernández**: His contributions were pivotal in Senegal’s advancement to the knockout stage, adding another compelling narrative of underdog success.
- **Group Stage Performance**: Senegal’s journey through the group stages was notable for its resilience and skill.

### Iran and Saudi Arabia
- **Iran's Participation Amidst Domestic Protests**: Iran's participation amid domestic protests highlighted the complex political context surrounding international sports events.
- **Saudi Arabia's Knockout Stage Advance**: Their surprising advancement to the knockout stage due to goal difference added another layer of intrigue to the tournament’s narrative.

## Broader Themes

### Team Performances
- **Brazil and Uruguay**: Their journeys through the group stages provided insights into the strengths and weaknesses of these traditional powerhouses.
- **Group B Highlights**: Key events such as Iran's participation amid domestic protests and Saudi Arabia's surprising advancement added depth to the tournament’s narrative.

### Individual Achievements
- **Kylian Mbappé and Enzo Fernández**: Their individual performances were central to their teams' success, showcasing the impact of key players on the outcome of matches.

### Controversies and Issues
- **Labor Issues and Corruption Allegations**: These broader themes highlighted the challenges faced by Qatar as a host nation, including labor rights concerns and corruption allegations.
- **Security Concerns**: Tensions between Qatar's security forces and media organizations, particularly with Iran International, raised questions about freedom of press and political context.

### LGBTQ+ Rights
- **Inclusion and Representation**: The tournament saw increased efforts to promote LGBTQ+ rights in football, reflecting broader societal changes and inclusivity movements.

## Conclusion

The 2022 FIFA World Cup was a multifaceted event with several key storylines and themes. From the dominant performances of France and Argentina to the impressive journey of Senegal, the tournament offered a rich tapestry of narratives that captured global attention. The broader themes of team performances, individual achievements, and controversies added depth to the overall narrative, making it a memorable edition of the World Cup.

While some reports provided specific insights into certain aspects, a comprehensive understanding required synthesizing multiple perspectives to capture the full scope of the 2022 FIFA World Cup.

---
## Comparison — Naive RAG Baseline

To appreciate what GraphRAG adds, we compare it against **naive retrieval-augmented
generation**: retrieve the top-3 most similar text chunks from the original PDF using
TF-IDF, then ask the LLM to answer the question from those chunks alone.

This is the approach standard RAG systems use. It works well for *specific* factual
questions ("Who scored the winning goal?") but struggles with *global* questions like
ours, because no single chunk captures the overall narrative (§1 of the paper).

> **Why TF-IDF and not embeddings?**  
> Embeddings would need a separate model and inference infrastructure. TF-IDF demonstrates
> the same fundamental limitation — surface-level keyword matching — which is the
> contrast the paper argues against. The point is not the retrieval method; it is the
> lack of cross-document synthesis.

In [9]:
# Re-create chunks from the PDF (same pipeline as Notebook 00, Step 1)
doc = fitz.open(PDF_PATH)
raw_text = "\n".join(page.get_text() for page in doc)
clean_text = re.sub(r"\[[\w,\s]+\]", "", raw_text)
clean_text = re.sub(r"\s{2,}", " ", clean_text)

splitter = RecursiveCharacterTextSplitter(chunk_size=2400, chunk_overlap=400)
chunks = splitter.split_text(clean_text)
print(f"Chunks: {len(chunks)}")

# TF-IDF retrieval
TOP_K = 3
vectorizer = TfidfVectorizer(stop_words="english")
chunk_matrix = vectorizer.fit_transform(chunks)
query_vec    = vectorizer.transform([QUESTION])
scores       = cosine_similarity(query_vec, chunk_matrix)[0]
top_indices  = scores.argsort()[::-1][:TOP_K]

print(f"Top-{TOP_K} chunks by TF-IDF similarity:")
for rank, idx in enumerate(top_indices, 1):
    print(f"  #{rank}  chunk[{idx}]  score={scores[idx]:.3f}  preview: {chunks[idx][:80]}...")

Chunks: 27
Top-3 chunks by TF-IDF similarity:
  #1  chunk[0]  score=0.180  preview: ← 2018
2026 →
2022 FIFA World Cup
2022 القدم لكرة العالم كأس
Tournament details
...
  #2  chunk[4]  score=0.169  preview: proceedings, and Indonesia's bid was rejected by FIFA in February 2010 after
the...
  #3  chunk[3]  score=0.153  preview: FIFA confirmed the group stage venue and kick-off times on 1 April 2022, followi...


In [10]:
NAIVE_RAG_PROMPT = """\
Answer the following question using only the provided text passages.
Style the response in markdown.

---Question---
{question}

---Passages---
{passages}

Output:"""

passages_text = "\n\n---\n\n".join(
    f"[Passage {rank+1}]\n{chunks[idx]}" for rank, idx in enumerate(top_indices)
)

naive_response = llm.invoke([
    HumanMessage(content=NAIVE_RAG_PROMPT.format(
        question=QUESTION,
        passages=passages_text,
    ))
])
naive_answer = strip_code_fence(naive_response.content)

display(Markdown("## Naive RAG Answer\n\n" + naive_answer))

## Naive RAG Answer

# Main Storylines and Themes of the 2022 FIFA World Cup

The 2022 FIFA World Cup, held from November 20 to December 18 in Qatar, featured several notable storylines and themes:

- **First World Cup in the Middle East**: This was the first time a World Cup took place in the Middle East and the Arabian Peninsula.
  
- **Historic Achievements**: Morocco made history by becoming the first African nation to reach the semi-finals of a World Cup. Argentina won their third title, their first since 1986.

- **Record-Breaking Performances**: Kylian Mbappé became the first player to score a hat-trick in a World Cup final and won the Golden Boot as he scored the most goals (eight) during the tournament.

- **Climate Considerations**: The event was held outside traditional months due to Qatar's extreme summer heat, taking place from November to December. This reduced the usual 32-team format to 64 matches played over a shorter period of 29 days.

- **Cost and Controversy**: The tournament cost an estimated $220 billion, making it the most expensive World Cup ever held. It faced criticism related to corruption scandals and operational risks associated with hosting in Qatar.

These elements combined to make the 2022 FIFA World Cup a significant event both on and off the field.

---
## Side-by-Side Comparison

In [11]:
separator = "\n\n---\n\n"

display(Markdown(
    "## GraphRAG Answer\n"
    "*Source: Map-Reduce over 15 community summaries*\n\n"
    + graphrag_answer
    + separator
    + "## Naive RAG Answer\n"
    "*Source: TF-IDF top-3 chunks from the raw PDF*\n\n"
    + naive_answer
))

## GraphRAG Answer
*Source: Map-Reduce over 15 community summaries*

# Main Storylines and Themes of the 2022 FIFA World Cup

## Overview

The 2022 FIFA World Cup, held in Qatar from November 20 to December 18, 2022, was marked by several key storylines and themes that captured global attention. This report synthesizes insights from various analyst reports to provide a comprehensive overview of the main narratives and themes.

## Key Storylines

### France's Dominance
- **Kylian Mbappé**: The French striker's exceptional goal-scoring record was a central theme, with his performances contributing significantly to France’s success.
- **Golden Ball Award**: Lionel Messi winning the Golden Ball for best player in the tournament highlighted Argentina's triumph and added another layer of narrative.

### Argentina's Triumph
- **Lionel Messi**: His performance and the award he received underscored the emotional journey and legacy of one of football's greatest players.
- **Final Match**: The final match between France and Argentina was a defining moment, with Messi winning the Golden Ball and leading his team to victory.

## Notable Performances

### Senegal's Impressive Journey
- **Enzo Fernández**: His contributions were pivotal in Senegal’s advancement to the knockout stage, adding another compelling narrative of underdog success.
- **Group Stage Performance**: Senegal’s journey through the group stages was notable for its resilience and skill.

### Iran and Saudi Arabia
- **Iran's Participation Amidst Domestic Protests**: Iran's participation amid domestic protests highlighted the complex political context surrounding international sports events.
- **Saudi Arabia's Knockout Stage Advance**: Their surprising advancement to the knockout stage due to goal difference added another layer of intrigue to the tournament’s narrative.

## Broader Themes

### Team Performances
- **Brazil and Uruguay**: Their journeys through the group stages provided insights into the strengths and weaknesses of these traditional powerhouses.
- **Group B Highlights**: Key events such as Iran's participation amid domestic protests and Saudi Arabia's surprising advancement added depth to the tournament’s narrative.

### Individual Achievements
- **Kylian Mbappé and Enzo Fernández**: Their individual performances were central to their teams' success, showcasing the impact of key players on the outcome of matches.

### Controversies and Issues
- **Labor Issues and Corruption Allegations**: These broader themes highlighted the challenges faced by Qatar as a host nation, including labor rights concerns and corruption allegations.
- **Security Concerns**: Tensions between Qatar's security forces and media organizations, particularly with Iran International, raised questions about freedom of press and political context.

### LGBTQ+ Rights
- **Inclusion and Representation**: The tournament saw increased efforts to promote LGBTQ+ rights in football, reflecting broader societal changes and inclusivity movements.

## Conclusion

The 2022 FIFA World Cup was a multifaceted event with several key storylines and themes. From the dominant performances of France and Argentina to the impressive journey of Senegal, the tournament offered a rich tapestry of narratives that captured global attention. The broader themes of team performances, individual achievements, and controversies added depth to the overall narrative, making it a memorable edition of the World Cup.

While some reports provided specific insights into certain aspects, a comprehensive understanding required synthesizing multiple perspectives to capture the full scope of the 2022 FIFA World Cup.

---

## Naive RAG Answer
*Source: TF-IDF top-3 chunks from the raw PDF*

# Main Storylines and Themes of the 2022 FIFA World Cup

The 2022 FIFA World Cup, held from November 20 to December 18 in Qatar, featured several notable storylines and themes:

- **First World Cup in the Middle East**: This was the first time a World Cup took place in the Middle East and the Arabian Peninsula.
  
- **Historic Achievements**: Morocco made history by becoming the first African nation to reach the semi-finals of a World Cup. Argentina won their third title, their first since 1986.

- **Record-Breaking Performances**: Kylian Mbappé became the first player to score a hat-trick in a World Cup final and won the Golden Boot as he scored the most goals (eight) during the tournament.

- **Climate Considerations**: The event was held outside traditional months due to Qatar's extreme summer heat, taking place from November to December. This reduced the usual 32-team format to 64 matches played over a shorter period of 29 days.

- **Cost and Controversy**: The tournament cost an estimated $220 billion, making it the most expensive World Cup ever held. It faced criticism related to corruption scandals and operational risks associated with hosting in Qatar.

These elements combined to make the 2022 FIFA World Cup a significant event both on and off the field.

---
## Summary

| Paper section | What we did |
|---|---|
| § 3.1.6 | Posed a global sensemaking question that no single chunk can answer |
| App. E.4 (Map) | Scored each community summary for helpfulness (0–100) |
| App. E.4 (Map) | Filtered score=0 communities, ranked the rest |
| App. E.4 (Reduce) | Synthesised ranked partial answers into a final markdown response |
| § 1 | Demonstrated the contrast with naive RAG (TF-IDF top-3 chunks) |

### What to look for in the comparison

- **GraphRAG** should cover multiple distinct themes: the Argentina–France final, the
  Qatar infrastructure, the FIFA corruption controversies, the qualification stories.
- **Naive RAG** will anchor on whichever chunks happened to contain the words from the
  question, likely returning a narrower, more factual paragraph.

This is the core argument of the paper (§1): *"queries that require an understanding of
an entire text corpus cannot be effectively addressed by local retrieval alone."*